In [4]:
pip install ucimlrepo


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# 1. Введение

В этом ноутбуке решается задача бинарной классификации на датасете **Adult Income**: по социально-демографическим признакам нужно предсказать, превышает ли доход человека порог **`>50K`** или нет (**`<=50K`**).

Цель baseline-подхода — построить простое, аккуратное и воспроизводимое решение, которое задаёт нижнюю разумную планку качества перед более сложными моделями. В качестве основной метрики используется **F1-score**.

Метрика выбрана не случайно: в датасете присутствует дисбаланс классов, поэтому accuracy не отражает реальное качество модели. Accuracy может оставаться высокой даже в случае, когда модель почти всегда предсказывает только большинство. **F1-score** лучше подходит для такой постановки, потому что одновременно учитывает precision и recall для положительного класса `>50K`.

## 2. Импорт и воспроизводимость

Ниже импортируются необходимые библиотеки, фиксируется `RANDOM_STATE = 42` и задаются минимальные настройки отображения. Это делает эксперимент воспроизводимым и упрощает чтение результатов.

Для корректного запуска сверху вниз в окружении должны быть установлены `ucimlrepo`, `pandas`, `numpy` и `scikit-learn`. 

In [5]:
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)

## 3. Загрузка данных

Датасет загружается **через `ucimlrepo`**, без использования `read_csv`. После загрузки признаки и целевая переменная объединяются в один DataFrame `df`.

В данных присутствуют и числовые, и категориальные признаки. Целевая переменная — `income`.

In [6]:
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=2)

X = adult.data.features.copy()
y = adult.data.targets.copy()

df = pd.concat([X, y], axis=1)

df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

print(f"Dataset shape: {df.shape}")
display(df.head())
df.info()

Dataset shape: (48842, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             48842 non-null  int64
 1   workclass       47879 non-null  str  
 2   fnlwgt          48842 non-null  int64
 3   education       48842 non-null  str  
 4   education_num   48842 non-null  int64
 5   marital_status  48842 non-null  str  
 6   occupation      47876 non-null  str  
 7   relationship    48842 non-null  str  
 8   race            48842 non-null  str  
 9   sex             48842 non-null  str  
 10  capital_gain    48842 non-null  int64
 11  capital_loss    48842 non-null  int64
 12  hours_per_week  48842 non-null  int64
 13  native_country  48568 non-null  str  
 14  income          48842 non-null  str  
dtypes: int64(6), str(9)
memory usage: 5.6 MB


## 4. Минимальный EDA

Сначала выполняется базовая очистка строковых признаков:

- убираются лишние пробелы через `str.strip()`;
- значения `"?"` переводятся в `NaN`;
- у целевой переменной удаляется точка в записях вида `>50K.`.

После этого можно корректно проверить пропуски, разделить признаки на числовые и категориальные, а также посмотреть распределение `income`.

Для этого датасета дисбаланс выражен достаточно явно: около **76%** наблюдений относятся к классу `<=50K`, и только около **24%** к классу `>50K`. В такой ситуации accuracy хуже как основная метрика: модель может показать формально высокий процент правильных ответов, просто угадывая большинство.

**F1-score** лучше подходит для baseline, потому что одновременно учитывает precision и recall именно по положительному классу `>50K` и не позволяет скрыть слабое качество за счёт доминирующего класса `<=50K`.

In [7]:
object_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()

df[object_columns] = df[object_columns].apply(lambda column: column.str.strip())
df[object_columns] = df[object_columns].replace("?", np.nan)
df["income"] = df["income"].str.replace(".", "", regex=False)

missing_values = (
    df.isna().sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)
missing_values = missing_values[missing_values["missing_count"] > 0]

if missing_values.empty:
    missing_values = pd.DataFrame(
        {"missing_count": [0]},
        index=["no_missing_values"],
    )

feature_df = df.drop(columns="income")
numeric_features = feature_df.select_dtypes(include="number").columns.tolist()
categorical_features = feature_df.select_dtypes(exclude="number").columns.tolist()

income_distribution = (
    df["income"]
    .value_counts(dropna=False)
    .rename_axis("income")
    .to_frame("count")
)
income_distribution["share"] = (
    income_distribution["count"] / income_distribution["count"].sum()
).round(4)
income_distribution = income_distribution.reindex(["<=50K", ">50K"])

feature_type_summary = pd.DataFrame(
    {
        "feature_type": ["numeric", "categorical"],
        "count": [len(numeric_features), len(categorical_features)],
    }
)

print("Missing values by column:")
display(missing_values)

print("Feature type summary:")
display(feature_type_summary)

print("Income distribution:")
display(income_distribution)

Missing values by column:


,missing_count
occupation,2809
workclass,2799
native_country,857


Feature type summary:


,feature_type,count
0,numeric,6
1,categorical,8


Income distribution:


,count,share
income,,
<=50K,37155,0.7607
>50K,11687,0.2393


## 5. Подготовка данных

На этом шаге формируются `X` и `y` для обучения модели. Целевая переменная `income` приводится к единому бинарному виду: `<=50K -> 0`, `>50K -> 1`.

Такое преобразование нужно, чтобы модель и метрика работали с целевым признаком единообразно и без неоднозначности в обозначении классов. После преобразования положительный класс `1` соответствует `>50K`, значит именно для него и интерпретируется `F1-score`.

Категориальные признаки требуют кодирования, потому что **логистическая регрессия не работает напрямую с категориальными признаками** в текстовом виде.

In [8]:
target_mapping = {"<=50K": 0, ">50K": 1}

X = feature_df.copy()
y = df["income"].map(target_mapping)

if y.isna().any():
    unexpected_labels = sorted(df.loc[y.isna(), "income"].dropna().unique())
    raise ValueError(f"Unexpected target labels: {unexpected_labels}")

y = y.astype(int)
print("Target distribution after mapping:")
print(y.value_counts().sort_index())
print(f"Positive class share: {y.mean():.3f}")

Target distribution after mapping:
income
0    37155
1    11687
Name: count, dtype: int64
Positive class share: 0.239


## 6. Train/test split

Данные делятся на обучающую и тестовую выборки в пропорции `80/20`.

Использование `stratify=y` важно, потому что оно сохраняет долю классов в обеих выборках и делает сравнение честным при дисбалансе.

Параметр `random_state=42` нужен для воспроизводимости: при повторном запуске получится то же самое разбиение. Наличие отдельного test set необходимо, чтобы оценивать качество модели на ранее невидимых данных.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "dataset": ["train", "test"],
        "rows": [X_train.shape[0], X_test.shape[0]],
        "positive_rate": [y_train.mean(), y_test.mean()],
    }
)

display(split_summary)

,dataset,rows,positive_rate
0,train,39073,0.239270
1,test,9769,0.239328


## 7. Dummy baseline

В качестве нижней границы качества используется `DummyClassifier(strategy="most_frequent")`. Такая модель всегда предсказывает самый частый класс и игнорирует реальные закономерности в признаках.

Если основная baseline-модель не превосходит такой подход, значит признаки либо плохо используются, либо модель не извлекает полезный сигнал из данных. Параметр `zero_division=0` в расчёте F1 здесь осмыслен: dummy-модель может вообще не предсказать ни одного объекта класса `>50K`, и тогда корректное значение F1 для положительного класса равно 0.

In [10]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train, y_train)

dummy_pred = dummy_model.predict(X_test)
dummy_f1 = f1_score(y_test, dummy_pred, zero_division=0)

print(f"Dummy F1-score: {dummy_f1:.4f}")

Dummy F1-score: 0.0000


## 8. Основная baseline-модель

В качестве основной baseline-модели используется **`LogisticRegression`**. Это простая, интерпретируемая baseline-модель, которая часто даёт сильную отправную точку для табличной бинарной классификации.

Предобработка выполняется **строго внутри `Pipeline`**, что исключает утечку данных: все импьютеры, масштабирование и кодирование обучаются только на train-части.

Почему именно такие шаги предобработки:

- `SimpleImputer(strategy="median")` для числовых признаков: медиана устойчива к выбросам.
- `StandardScaler()` для числовых признаков: модель чувствительна к масштабу признаков.
- `SimpleImputer(strategy="most_frequent")` для категориальных признаков: сохраняет наиболее вероятное значение.
- `OneHotEncoder(handle_unknown="ignore")`: логистическая регрессия не работает напрямую с категориальными признаками, поэтому их нужно кодировать. Параметр `handle_unknown="ignore"` защищает от ошибок на новых категориях в тестовой выборке.
- `solver="liblinear"` выбран как устойчивый вариант для бинарной логистической регрессии в baseline-задаче; этого достаточно для простой и интерпретируемой стартовой модели.

In [11]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

logreg_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
                solver="liblinear",
            ),
        ),
    ]
)

logreg_model.fit(X_train, y_train)

logreg_pred = logreg_model.predict(X_test)
logreg_f1 = f1_score(y_test, logreg_pred, zero_division=0)

print(f"LogisticRegression F1-score: {logreg_f1:.4f}")

LogisticRegression F1-score: 0.6558


## 9. Сравнение моделей

Сведём результаты в одну таблицу и сравним dummy baseline с логистической регрессией по **F1-score**.

In [12]:
results = pd.DataFrame(
    {
        "model": ["Dummy", "LogisticRegression"],
        "f1_score": [dummy_f1, logreg_f1],
    }
)

results["f1_score"] = results["f1_score"].round(4)
f1_gain = logreg_f1 - dummy_f1

display(results)
print(f"Absolute F1 gain over Dummy: {f1_gain:.4f}")

,model,f1_score
0,Dummy,0.0000
1,LogisticRegression,0.6558


Absolute F1 gain over Dummy: 0.6558


По таблице выше видно, что `LogisticRegression` превосходит `DummyClassifier` по F1-score с заметным запасом. Это означает, что признаки действительно содержат полезный сигнал, а baseline-модель уже умеет извлекать из них информацию лучше, чем простое предсказание самого частого класса.

Здесь важно интерпретировать не только сам факт превосходства, но и его смысл: `DummyClassifier` ориентируется только на большинство и фактически игнорирует класс `>50K`, тогда как `LogisticRegression` уже на baseline-уровне находит часть объектов положительного класса. Следовательно, построенный baseline не только задаёт нижнюю планку качества, но и подтверждает, что дальнейшее улучшение модели имеет смысл.

## 10. Итог

В ходе работы был построен и оценён корректный baseline для задачи бинарной классификации на датасете **Adult Income**. Данные были загружены через `ucimlrepo`, объединены в единый DataFrame и приведены к пригодному для моделирования виду: обработаны пропуски, удалены лишние пробелы и унифицирована запись целевой переменной `income`.

После этого было выполнено стратифицированное разделение на обучающую и тестовую выборки, что позволило сохранить исходное соотношение классов. В качестве нижней границы качества рассмотрен `DummyClassifier`, предсказывающий наиболее частый класс. Затем была обучена baseline-модель `LogisticRegression`, для которой вся предобработка выполнена строго внутри `Pipeline`: числовые признаки заполнялись медианой и масштабировались, а категориальные заполнялись наиболее частым значением и кодировались с помощью `OneHotEncoder`.

Сравнение моделей по **F1-score** показало, что логистическая регрессия существенно превосходит константный baseline. Это означает, что признаки действительно содержат полезный сигнал, а выбранная схема предобработки позволяет модели корректно использовать как числовые, так и категориальные данные без утечки информации из тестовой выборки.

Таким образом, поставленная цель достигнута: получено аккуратное, воспроизводимое и интерпретируемое baseline-решение, которое задаёт разумную отправную точку для дальнейшего улучшения качества модели.